## Transformers (HuggingFace)

https://huggingface.co/models

In [1]:
from transformers import pipeline

/Users/sebastiancg/Developer/scaceresg/ai-engineering/notebooks/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
classifier = pipeline("sentiment-analysis")

result = classifier("I've been waiting for a Hugging Face course my whole life.")

result

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 8834.41it/s]


[{'label': 'POSITIVE', 'score': 0.9982948899269104}]

In [5]:
ner = pipeline("ner", model="dslim/bert-base-NER")

result = ner("My name is Wolfgang and I live in Berlin")

result

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10141.63it/s]
BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity': 'B-PER',
  'score': np.float32(0.9990139),
  'index': 4,
  'word': 'Wolfgang',
  'start': 11,
  'end': 19},
 {'entity': 'B-LOC',
  'score': np.float32(0.999645),
  'index': 9,
  'word': 'Berlin',
  'start': 34,
  'end': 40}]

In [2]:
zeroshot_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

sequence_to_classify = "One day I will see the world"

candidate_labels = ["travel", "cooking", "dancing"]

result = zeroshot_classifier(
    sequence_to_classify,
    candidate_labels=candidate_labels,
)

result


Loading weights: 100%|██████████| 515/515 [00:00<00:00, 3816.68it/s]


{'sequence': 'One day I will see the world',
 'labels': ['travel', 'dancing', 'cooking'],
 'scores': [0.9941016435623169, 0.003126119961962104, 0.002772232750430703]}

## HuggingFace with Pytorch/Tensorflow

In [8]:
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
## Classification
sentence = "I've been waiting for a Hugging Face course my whole life."

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

inputs = tokenizer(sentence, return_tensors="pt")

inputs

{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662,  2227,
          2607,  2026,  2878,  2166,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [9]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")


with torch.no_grad():
    logits = model(**inputs).logits

predictions = logits.argmax().item()

model.config.id2label[predictions]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 20012.28it/s]


'POSITIVE'

## Saving and Loading Models

In [ ]:
model_directory = "my_models"

## Saving models
tokenizer.save_pretrained(model_directory)
model.save_pretrained(model_directory)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.42it/s]


In [11]:
## Loading models
my_tokenizer = AutoTokenizer.from_pretrained(model_directory)
my_model = AutoModelForSequenceClassification.from_pretrained(model_directory)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 8440.22it/s]


## BERT

In [12]:
import torch

from transformers import BertForQuestionAnswering, BertTokenizer

In [13]:
model_name = "bert-large-uncased-whole-word-masking-finetuned-squad"

model = BertForQuestionAnswering.from_pretrained(model_name)

tokenizer = BertTokenizer.from_pretrained(model_name)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5210.15it/s]
BertForQuestionAnswering LOAD REPORT from: bert-large-uncased-whole-word-masking-finetuned-squad
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Embeddings

In [14]:
# Embeddings
question = "When was the first dvd released?"
answer_text = "The first DVD (Digital Versatile Disc) was released on March 24, 1997. It was a movie titled 'Twister' and was released in Japan. DVDs quickly gained popularity as a replacement for VHS tapes and became a common format for storing and distributing digital video and data."

In [17]:
encoding = tokenizer(text=question, text_pair=answer_text)
encoding

{'input_ids': [101, 2043, 2001, 1996, 2034, 4966, 2207, 1029, 102, 1996, 2034, 4966, 1006, 3617, 22979, 5860, 1007, 2001, 2207, 2006, 2233, 2484, 1010, 2722, 1012, 2009, 2001, 1037, 3185, 4159, 1005, 9792, 2121, 1005, 1998, 2001, 2207, 1999, 2900, 1012, 22477, 2855, 4227, 6217, 2004, 1037, 6110, 2005, 17550, 13324, 1998, 2150, 1037, 2691, 4289, 2005, 23977, 1998, 20083, 3617, 2678, 1998, 2951, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [22]:
inputs = encoding.get("input_ids")
sentence_embedding = encoding.get("token_type_ids")

tokens = tokenizer.convert_ids_to_tokens(inputs)

In [23]:
tokenizer.decode(101)

'[CLS]'

In [24]:
tokenizer.decode(102)

'[SEP]'

In [25]:
output = model(input_ids=torch.tensor([inputs]), token_type_ids=torch.tensor([sentence_embedding]))

In [26]:
output.keys()

odict_keys(['start_logits', 'end_logits'])

### BERT Chatbot

In [27]:
sunset_motors_context = "Sunset Motors is a renowned automobile dealership that has been a cornerstone of the automotive industry since its establishment in 1978. Located in the picturesque town of Crestwood, nestled in the heart of California's scenic Central Valley, Sunset Motors has built a reputation for excellence, reliability, and customer satisfaction over the past four decades. Founded by visionary entrepreneur Robert Anderson, Sunset Motors began as a humble, family-owned business with a small lot of used cars. However, under Anderson's leadership and commitment to quality, it quickly evolved into a thriving dealership offering a wide range of vehicles from various manufacturers. Today, the dealership spans over 10 acres, showcasing a vast inventory of new and pre-owned cars, trucks, SUVs, and luxury vehicles. One of Sunset Motors' standout features is its dedication to sustainability. In 2010, the dealership made a landmark decision to incorporate environmentally friendly practices, including solar panels to power the facility, energy-efficient lighting, and a comprehensive recycling program. This commitment to eco-consciousness has earned Sunset Motors recognition as an industry leader in sustainable automotive retail. Sunset Motors proudly offers a diverse range of vehicles, including popular brands like Ford, Toyota, Honda, Chevrolet, and BMW, catering to a wide spectrum of tastes and preferences. In addition to its outstanding vehicle selection, Sunset Motors offers flexible financing options, allowing customers to secure affordable loans and leases with competitive interest rates."
print(sunset_motors_context)

Sunset Motors is a renowned automobile dealership that has been a cornerstone of the automotive industry since its establishment in 1978. Located in the picturesque town of Crestwood, nestled in the heart of California's scenic Central Valley, Sunset Motors has built a reputation for excellence, reliability, and customer satisfaction over the past four decades. Founded by visionary entrepreneur Robert Anderson, Sunset Motors began as a humble, family-owned business with a small lot of used cars. However, under Anderson's leadership and commitment to quality, it quickly evolved into a thriving dealership offering a wide range of vehicles from various manufacturers. Today, the dealership spans over 10 acres, showcasing a vast inventory of new and pre-owned cars, trucks, SUVs, and luxury vehicles. One of Sunset Motors' standout features is its dedication to sustainability. In 2010, the dealership made a landmark decision to incorporate environmentally friendly practices, including solar p

In [30]:
def faq_bot(question: str):
    """Chatbot that answers questions about Sunset Motors using a BERT model."""
    
    context = sunset_motors_context
    # Tokenize the question and context
    input_ids = tokenizer.encode(text=question, text_pair=context)
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    sep_index = input_ids.index(tokenizer.sep_token_id)
    num_seg_a = sep_index + 1
    num_seg_b = len(input_ids) - num_seg_a
    segment_ids = [0] * num_seg_a + [1] * num_seg_b
    
    output = model(input_ids=torch.tensor([input_ids]), token_type_ids=torch.tensor([segment_ids]))
    answer_start = torch.argmax(output.start_logits)
    answer_end = torch.argmax(output.end_logits)

    if answer_end >= answer_start:
        answer = " ".join(tokens[answer_start:answer_end+1])
    else:
        answer = "I don't know how to answer that question. Can you please rephrase it?"

    corrected_answer = ""
    for word in answer.split():
        if word[0:2] == "##":
            corrected_answer += word[2:]
        else:
            corrected_answer += " " + word
    
    return corrected_answer

In [31]:
faq_bot("What is the name of the owner of Sunset Motors?")

' robert anderson'

In [32]:
faq_bot("Where is the dealership located?")

' crestwood'

In [33]:
faq_bot("What make of cars does Sunset Motors sell?")

' ford , toyota , honda , chevrolet , and bmw'

In [34]:
faq_bot("How large is the dealership?")

' 10 acres'